In [93]:
import numpy as np
import scipy.stats as stats
from scipy.optimize import root
from pathlib import Path
from const import DISTANCE, N_PART, PIXELS
from beta_weights import load_params, create_voters
from scipy.optimize import minimize

In [94]:
def create_medians(n):
    return np.arange(0.5+1/n/2, 1, 1/n)

In [95]:
def create_weights(median, D_target, X0):
    assert median > 0.5, "Median must be greater than 0.5"

    def system_of_equations(x):
        a, b = x[0], x[1]

        if a < 0 or b < 0:
            return [1e10, 1e10]

        cdf_ab = stats.beta.cdf(x=median, a=a, b=b)
        cdf_a1b = stats.beta.cdf(x=median, a=a + 1, b=b)
       
        eq1 = cdf_ab - 0.5
        ex = a / (a + b)
        eq2 = ex * (1 - 2 * cdf_a1b) - D_target
           
        return [eq1, eq2]

    sol = root(system_of_equations, X0, tol=1e-14, method='lm')
    a_sol, b_sol = sol.x[0], sol.x[1]
    return (a_sol, b_sol)

def generate_param(
    pixels: int,
    distance: float = DISTANCE
) -> np.ndarray:
    medians = create_medians(pixels)
    N_medians = len(medians)
    X0 = [1.0, 1.0]
    params = np.empty((N_medians, 2), dtype=np.float64)
    for i, m in enumerate(medians):
        (a, b) = create_weights(m, distance, X0)
        X0 = [a, b]
        params[i] = (a, b)

    mirrored_params = np.flip(params, axis=0)[:, ::-1]
    params = np.concatenate((mirrored_params, params), axis=0)
    return params

In [96]:
PIXELS = 500
voters = create_voters(16)
params = generate_param(PIXELS, DISTANCE)[PIXELS//2:, :]
medians = create_medians(PIXELS)
DISTANCE = 0.3
n_part = 16

voters = create_voters(n_part)
voters_w = create_voters(n_part, with_edge=True)
medians = create_medians(PIXELS)

TypeError: create_voters() got an unexpected keyword argument 'with_edge'

In [ ]:
def cdf_interval(voters, a, b):
    lower = stats.beta.cdf(x=voters - 1 / len(voters) / 2, a=a, b=b)
    upper = stats.beta.cdf(x=voters + 1 / len(voters) / 2, a=a, b=b)
    return upper - lower

def discrete_median(w1, a, b):
    probs_w = cdf_edge(w1, a, b)
    cumulative_from_right = np.cumsum(probs_w[::-1])
    index_reversed = np.searchsorted(
        cumulative_from_right,
        0.5,
        side="left"
    )
    index = len(probs_w) - 1 - index_reversed
    return voters_w[index]

def discrete_deviation(w1, a, b):
    probs_w = cdf_edge(w1, a, b)
    return np.sum(probs_w*np.abs(discrete_median(w1, a, b)-voters_w))

def cdf_edge(w1, a, b):
    probs = cdf_interval(voters, a=a, b=b)
    probs_w = np.concatenate(([0.0], probs, [w1]))
    return (probs_w/probs_w.sum())

In [108]:
def create_voters(n_part, with_edge = False):
    voters = np.arange(1 / n_part / 2, 1, 1 / n_part)
    if with_edge:
        return np.concatenate(([0.0], voters, [1.0]))
    return voters

def create_edge_weights(median, n_part, D_target, params_w0):
    assert median > 0.5, "Median must be greater than 0.5"

    def system_eq(params_w):
        a, b, w1 = params_w

        if w1 < 0 or w1 >= 1:
            return 1e10

        if a < 1 or b < 1:
            return 1e10
        
        eq1 = discrete_median(w1, a, b)-median
        eq2 = discrete_deviation(w1, a, b)-D_target

        return eq1**2 + eq2**2

    result = minimize(
        system_eq,
        x0=np.asarray(params_w0, dtype=float),
        method="Nelder-Mead",
        options={
            "xatol": 1e-12,
            "fatol": 1e-12,
            "maxiter": 2000,
    })
    a, b, w1 = result.x
    print(f"for median {median}, discrete {discrete_median(w1, a, b)}")
    return w1, cdf_edge(w1, a, b)

In [114]:
create_edge_weights(medians[0], 16, DISTANCE, [1.7, 1.7, 0])

for median 0.501, discrete 0.53125


(np.float64(0.003055659949043996),
 array([0.        , 0.0623096 , 0.0623096 , 0.0623096 , 0.0623096 ,
        0.0623096 , 0.0623096 , 0.0623096 , 0.0623096 , 0.0623096 ,
        0.0623096 , 0.0623096 , 0.0623096 , 0.0623096 , 0.0623096 ,
        0.0623096 , 0.0623096 , 0.00304635]))

In [111]:
medians[0]

np.float64(0.501)

In [82]:
np.array([0.        , 0.12918763, 0.06899468, 0.05756584, 0.051844  ,
        0.04847201, 0.04639694, 0.04517863, 0.04461453, 0.04462257,
        0.04520332, 0.04644016, 0.04853759, 0.05193929, 0.0577064 ,
        0.06922328, 0.13000645]).sum()

np.float64(0.98593332)

In [149]:
Ws = [0, 0]
for m, p in zip(medians, params):
    (w0, w1), _  = create_edge_weights(m, p, 16, DISTANCE, Ws)
    Ws = [w0, w1]

for median 0.501, discrete 0.53125
for median 0.503, discrete 0.53125
for median 0.505, discrete 0.53125
for median 0.507, discrete 0.53125
for median 0.509, discrete 0.53125
for median 0.511, discrete 0.53125
for median 0.513, discrete 0.53125
for median 0.515, discrete 0.53125
for median 0.517, discrete 0.53125
for median 0.519, discrete 0.53125
for median 0.521, discrete 0.53125
for median 0.523, discrete 0.53125
for median 0.525, discrete 0.53125
for median 0.527, discrete 0.53125
for median 0.529, discrete 0.53125
for median 0.531, discrete 0.53125
for median 0.533, discrete 0.53125
for median 0.535, discrete 0.53125
for median 0.537, discrete 0.53125
for median 0.539, discrete 0.53125
for median 0.541, discrete 0.53125
for median 0.543, discrete 0.53125
for median 0.545, discrete 0.53125
for median 0.547, discrete 0.53125
for median 0.549, discrete 0.53125
for median 0.551, discrete 0.53125
for median 0.553, discrete 0.53125
for median 0.555, discrete 0.53125
for median 0.557, di

In [81]:
np.array([abs(discrete_median(a, b)-stats.beta.ppf(0.5, a, b)) 
for i, (a, b) in enumerate(params)]).max()

np.float64(0.03124999999999968)

In [31]:
def make_weigts(a, b, n_part):
    voters = create_voters(n_part)
    weights = cdf_interval(voters, a, b)
    

In [6]:
PIXELS = 1000
DISTANCE = 0.4
medians = create_medians(PIXELS)

In [7]:
old_params = generate_params_old(PIXELS, DISTANCE)[PIXELS//2:, :]
diffs = np.array([abs(stats.beta.ppf(0.5, a, b) - m) for (a, b), m in zip(old_params, medians)])
diffs.max()

np.float64(2.1094237467877974e-15)